# 04 Gold — card meta metrics

The report layer. Aggregates the silver card-grain table and quality check
table into two small, dashboard-ready Delta tables:

- **`gold_card_metrics`** — one row per card: pick rate, win rate, sample size,
  enriched with `elixir_band` so the dashboard can cut by cost tier.
- **`gold_overview`** — a single KPI row for the dashboard's scalar tiles
  (totals, freshness, validity).

**Meta scope:** Trophy ladder only (`is_ladder = true`). Path of Legend and the
other modes are deliberately excluded so win/pick rates describe one metagame.

In [0]:
from pyspark.sql import functions as F, types as T

CATALOG, SCHEMA = "workspace", "clash"

# Sources
SILVER_BATTLES     = f"{CATALOG}.{SCHEMA}.silver_battles"
SILVER_DECK_CARDS  = f"{CATALOG}.{SCHEMA}.silver_deck_cards"
DQ_RESULTS         = f"{CATALOG}.{SCHEMA}.dq_results"
QUARANTINE_BATTLES = f"{CATALOG}.{SCHEMA}.quarantine_battles"

# Targets (gold)
GOLD_CARD_METRICS = f"{CATALOG}.{SCHEMA}.gold_card_metrics"
GOLD_OVERVIEW     = f"{CATALOG}.{SCHEMA}.gold_overview"
GOLD_MODE_BREAKDOWN = f"{CATALOG}.{SCHEMA}.gold_mode_breakdown"

SCOPE_LABEL = "trophy_ladder"   # is_ladder = true
print("scope:", SCOPE_LABEL)

scope: trophy_ladder


## `gold_card_metrics`

`silver_deck_cards` is already at card grain: one row per card, per side, per
battle. For this layer we scope it to trophy ladder mode only, but all the
modes are kept in the database for other extended analysis:

- **pick rate** = appearances of a card / total deck instances (each battle has
  two decks, so the denominator is `2 × ladder battles`, minus any missing decks).
- **win rate** = that card's side wins / its appearances. `won` is the side's
  outcome from crown counts; equal-crown draws (~0.1%) count as non-wins.

`n_appearances` is kept as a plain column so the dashboard can filter out
tiny-sample cards instead of needing a confidence interval.

In [0]:
# Card-grain rows, scoped to trophy ladder.
ladder_cards = spark.table(SILVER_DECK_CARDS).filter("is_ladder = true")

# Pick-rate denominator: distinct deck instances (battle_id x side) in scope.
total_decks = ladder_cards.select("battle_id", "side").distinct().count()
print("deck instances in scope:", total_decks)

gold_card = (ladder_cards
    .groupBy("card_id", "card_name", "rarity", "elixir_cost", "elixir_band")
    .agg(
        F.count("*").alias("n_appearances"),
        # count(won) ignores nulls, so the win-rate denominator only counts
        # battles whose outcome was decidable (crowns present on both sides).
        F.count("won").alias("n_decided"),
        F.sum(F.col("won").cast("int")).alias("wins"),
    )
    .withColumn("pick_rate", F.col("n_appearances") / F.lit(total_decks))
    .withColumn("win_rate", F.col("wins") / F.col("n_decided"))
    .withColumn("scope", F.lit(SCOPE_LABEL))
    .withColumn("generated_at", F.current_timestamp()))

(gold_card.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_CARD_METRICS))

gc = spark.table(GOLD_CARD_METRICS)
print(gc.count(), "cards in gold_card_metrics")
display(gc.orderBy(F.desc("pick_rate")).limit(10))

deck instances in scope: 191712
121 cards in gold_card_metrics


card_id,card_name,rarity,elixir_cost,elixir_band,n_appearances,n_decided,wins,pick_rate,win_rate,scope,generated_at
28000011,The Log,legendary,2,cheap,53937,53937,28010,0.2813438908362544,0.5193095648627102,trophy_ladder,2026-06-10T11:18:51.236Z
28000001,Arrows,common,3,mid,52667,52667,25971,0.27471937072275077,0.49311713217004954,trophy_ladder,2026-06-10T11:18:51.236Z
28000000,Fireball,rare,4,mid,45373,45373,23736,0.2366727174094475,0.5231304961100214,trophy_ladder,2026-06-10T11:18:51.236Z
28000008,Zap,common,2,cheap,38936,38936,19847,0.2030963111333667,0.5097339223340867,trophy_ladder,2026-06-10T11:18:51.236Z
26000021,Hog Rider,rare,4,mid,37608,37608,19085,0.19616925388082124,0.5074718145075516,trophy_ladder,2026-06-10T11:18:51.236Z
26000055,Mega Knight,legendary,7,heavy,37257,37257,17680,0.1943383825738608,0.4745416968623346,trophy_ladder,2026-06-10T11:18:51.236Z
26000000,Knight,common,3,mid,35533,35533,18039,0.18534572692371892,0.5076689274758674,trophy_ladder,2026-06-10T11:18:51.236Z
26000011,Valkyrie,rare,4,mid,34344,34344,16217,0.17914371557336004,0.47219310505474027,trophy_ladder,2026-06-10T11:18:51.236Z
28000015,Barbarian Barrel,epic,2,cheap,32236,32236,17679,0.16814805541645803,0.5484241220995161,trophy_ladder,2026-06-10T11:18:51.236Z
26000010,Skeletons,common,1,cheap,32029,32029,16749,0.16706831079953263,0.5229323425645509,trophy_ladder,2026-06-10T11:18:51.236Z


## `gold_overview`

One KPI row for the dashboard's scalar tiles. Counts come from silver;
freshness and validity are read from the latest `dq_results` run rather than
recomputed, so the quality layer stays the single source of truth.

- `distinct_participants` — every tag seen on either side. Mostly one-off
  opponents we never crawled, so it is far larger than the ~10k seed players.
- `validity_pct` — share of battles that passed every *error* check this run
  (i.e. were not quarantined).

In [0]:
sb = spark.table(SILVER_BATTLES)
total_battles  = sb.count()
ladder_battles = sb.filter("is_ladder = true").count()

# Distinct participants: union of both sides\' tags (dominated by one-off opponents).
participants = (sb.select(F.col("player_tag").alias("tag"))
    .union(sb.select(F.col("opponent_tag").alias("tag")))
    .filter(F.col("tag").isNotNull()).distinct().count())

distinct_cards = spark.table(GOLD_CARD_METRICS).count()

# Freshness + validity from the latest dq_results run (single source of truth).
latest_run = spark.table(DQ_RESULTS).agg(F.max("run_id")).first()[0]
dq = spark.table(DQ_RESULTS).filter(F.col("run_id") == latest_run)
freshness_hours = (dq.filter("check_name = 'freshness_under_24h'")
    .agg(F.max("metric_value")).first()[0])

quarantined = spark.table(QUARANTINE_BATTLES).select("battle_id").distinct().count()
validity_pct = 100.0 * (1 - quarantined / total_battles) if total_battles else None

overview_schema = T.StructType([
    T.StructField("total_battles", T.LongType()),
    T.StructField("ladder_battles", T.LongType()),
    T.StructField("distinct_participants", T.LongType()),
    T.StructField("distinct_cards", T.LongType()),
    T.StructField("freshness_hours", T.DoubleType()),
    T.StructField("validity_pct", T.DoubleType()),
    T.StructField("scope", T.StringType()),
    T.StructField("dq_run_id", T.StringType()),
])
overview = spark.createDataFrame([(
    total_battles, ladder_battles, participants, distinct_cards,
    float(freshness_hours) if freshness_hours is not None else None,
    float(validity_pct) if validity_pct is not None else None,
    SCOPE_LABEL, latest_run,
)], overview_schema).withColumn("generated_at", F.current_timestamp())

(overview.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_OVERVIEW))
display(spark.table(GOLD_OVERVIEW))

total_battles,ladder_battles,distinct_participants,distinct_cards,freshness_hours,validity_pct,scope,dq_run_id,generated_at
318176,95856,251250,121,54.49285443111111,91.63104696771597,trophy_ladder,20260610T102532,2026-06-10T11:18:58.816Z


## `gold_mode_breakdown`

Battle count and share of the corpus per game mode, across **all** modes (not
scoped to ladder). This is what justifies the trophy-ladder scope of the metric
tables — it shows how big the `Ladder` slice is relative to Path of Legend and
everything else. The `is_ladder` flag lets the dashboard highlight the in-scope
slice.

In [0]:
sb = spark.table(SILVER_BATTLES)
mode_total = sb.count()

gold_mode = (sb
    .groupBy("game_mode", "is_ladder")
    .agg(F.count("*").alias("n_battles"))
    .withColumn("pct_of_total", 100.0 * F.col("n_battles") / F.lit(mode_total))
    .withColumn("generated_at", F.current_timestamp())
    .orderBy(F.desc("n_battles")))

(gold_mode.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(GOLD_MODE_BREAKDOWN))

print(f"{spark.table(GOLD_MODE_BREAKDOWN).count()} game modes; "
      f"ladder share = "
      f"{spark.table(GOLD_MODE_BREAKDOWN).filter('is_ladder').agg(F.sum('pct_of_total')).first()[0]:.1f}%")
display(spark.table(GOLD_MODE_BREAKDOWN).limit(20))

43 game modes; ladder share = 30.1%


game_mode,is_ladder,n_battles,pct_of_total,generated_at
Ranked1v1_NewArena,false,122581,38.52616162124107,2026-06-10T11:19:03.887Z
Ladder,true,95856,30.126722317208085,2026-06-10T11:19:03.887Z
ClanWar_BoatBattle,false,20325,6.387973951523685,2026-06-10T11:19:03.887Z
Ranked1v1_NewArena2,false,12697,3.9905586845016594,2026-06-10T11:19:03.887Z
TripleElixir_Friendly,false,11554,3.631323544201951,2026-06-10T11:19:03.887Z
TripleElixir_Ladder,false,10825,3.4022050688926884,2026-06-10T11:19:03.887Z
CW_Battle_1v1,false,8887,2.793108216835965,2026-06-10T11:19:03.887Z
TeamVsTeam,false,6465,2.0318942974957257,2026-06-10T11:19:03.887Z
CW_Duel_1v1,false,6156,1.934778235944886,2026-06-10T11:19:03.887Z
Challenge_AllCards_EventDeck_NoSet,false,5640,1.7726038418988233,2026-06-10T11:19:03.887Z


## Dashboard preview — the two headline charts

Top cards by win rate (with a minimum-sample floor so a rarely-played card can't
spike the chart) and top cards by pick rate.

In [0]:
gc = spark.table(GOLD_CARD_METRICS)

MIN_APPEARANCES = 500   # floor out tiny-sample cards from the win-rate chart
print(f"top cards by win rate (>= {MIN_APPEARANCES} appearances)")
display(gc.filter(F.col("n_appearances") >= MIN_APPEARANCES)
    .orderBy(F.desc("win_rate"))
    .select("card_name", "rarity", "elixir_cost", "win_rate", "pick_rate", "n_appearances")
    .limit(10))

print("top cards by pick rate")
display(gc.orderBy(F.desc("pick_rate"))
    .select("card_name", "rarity", "elixir_cost", "pick_rate", "win_rate", "n_appearances")
    .limit(10))

top cards by win rate (>= 500 appearances)


card_name,rarity,elixir_cost,win_rate,pick_rate,n_appearances
Zappies,rare,4,0.557914827627341,0.04372704890669337,8383
Cannon Cart,epic,5,0.5525881470367592,0.03476569020196962,6665
Goblin Cage,rare,4,0.5524015414982612,0.0554947003839092,10639
Rascals,common,5,0.5512892692264385,0.04632469537639793,8881
Royal Ghost,legendary,3,0.5505124450951684,0.07125271240193624,13660
Barbarian Barrel,epic,2,0.5484241220995161,0.16814805541645803,32236
Mortar,common,4,0.5478600170710889,0.04277770822901018,8201
Skeleton Dragons,common,4,0.544,0.026080787848439327,5000
Bomb Tower,rare,4,0.5435416666666667,0.02503755633450175,4800
Royal Recruits,common,7,0.5406923312570203,0.05108183107995326,9793


top cards by pick rate


card_name,rarity,elixir_cost,pick_rate,win_rate,n_appearances
The Log,legendary,2,0.2813438908362544,0.5193095648627102,53937
Arrows,common,3,0.27471937072275077,0.49311713217004954,52667
Fireball,rare,4,0.2366727174094475,0.5231304961100214,45373
Zap,common,2,0.2030963111333667,0.5097339223340867,38936
Hog Rider,rare,4,0.19616925388082124,0.5074718145075516,37608
Mega Knight,legendary,7,0.1943383825738608,0.4745416968623346,37257
Knight,common,3,0.18534572692371892,0.5076689274758674,35533
Valkyrie,rare,4,0.17914371557336004,0.47219310505474027,34344
Barbarian Barrel,epic,2,0.16814805541645803,0.5484241220995161,32236
Skeletons,common,1,0.16706831079953263,0.5229323425645509,32029
